# Měkké přistání rakety

## Zadání slovy

> Raketa padá z výšky 1 500 m rychlostí přes 100 m/s a má přistát přesně na
> plošině — s nulovou rychlostí, aniž by proletěla pod zem nebo vybočila
> z přistávacího kužele. Motor nejde vypnout a znovu zapálit; smí jen škrtit
> mezi 5 a 30 kN. Paliva je málo, takže se ho má spotřebovat co nejméně.
>
> **Jak má počítač řídit tah motoru v každé z 60 vteřin sestupu?**

Není to školní vymyšlenost: přesně tuhle úlohu řeší algoritmus **G-FOLD**
(Açıkmeşe a Ploen), který běží na palubě přistávajících raket. Důvod, proč se
smí pustit v reálném čase pár set metrů nad zemí, je ten, že je **konvexní** —
solver doběhne v předvídatelném čase a nemůže „uvíznout“.

## Formulace

Čas se rozseká na $N = 60$ kroků po $\Delta t = 1$ s. Proměnné jsou polohy
$\mathbf p_k$, rychlosti $\mathbf v_k$ a vektory tahu $\mathbf T_k$ ve všech
krocích, plus pomocná $\sigma_k$ — dohromady přes 350 čísel.

$$
\begin{aligned}
\text{minimize}\quad & \textstyle\sum_k \sigma_k\,\Delta t && \text{palivo}\\
\text{subject to}\quad
& \mathbf v_{k+1}=\mathbf v_k+\Delta t\left(\tfrac{\mathbf T_k}{m}-\mathbf g\right)
  && \text{Newton}\\
& \mathbf p_{k+1}=\mathbf p_k+\Delta t\,\tfrac{\mathbf v_k+\mathbf v_{k+1}}{2}
  && \text{poloha z rychlosti}\\
& \mathbf p_0,\mathbf v_0\ \text{zadane},\qquad
  \mathbf p_N=\mathbf 0,\ \mathbf v_N=\mathbf 0 && \text{mekke pristani}\\
& p_{k,2}\ \ge\ \tfrac12\,|p_{k,1}| && \text{pristavaci kuzel}\\
& \lVert \mathbf T_k\rVert_2\le\sigma_k,\qquad
  T_{\min}\le\sigma_k\le T_{\max} && \text{skrceni motoru (N)}
\end{aligned}
$$

Data: $m = 2000$ kg, $\mathbf p_0 = (1200,\ 1500)$ m, $\mathbf v_0 = (-60,\ -80)$
m/s, $T_{\min} = 5$ kN, $T_{\max} = 30$ kN.

Poslední řádek je pointa pro celý předmět. Skutečné omezení motoru je
$\lVert\mathbf T_k\rVert \ge T_{\min}$ — a to je **nekonvexní**, protože zakazuje
malé tahy uprostřed přípustné koule. Zavedením pomocné proměnné $\sigma_k$, které
se meze předepíšou a norma se k ní přiváže shora, vznikne konvexní úloha (SOCP)
a dá se **dokázat, že optimum je totožné** (*lossless convexification*). Těžké
tedy nebylo úlohu vyřešit, ale zapsat.

## Od zadání ke kódu

| v zadání | v kódu |
|---|---|
| $\mathbf p_k,\ \mathbf v_k$ | `p = cp.Variable((N + 1, 2))`, `v = cp.Variable((N + 1, 2))` |
| $\mathbf T_k,\ \sigma_k$ | `T = cp.Variable((N, 2))`, `sigma = cp.Variable(N)` |
| Newton a poloha z rychlosti | smyčka `for k in range(N)` |
| měkké přistání | `p[N] == 0, v[N] == 0` |
| přistávací kužel | `p[:, 1] >= 0.5 * cp.abs(p[:, 0])` |
| $\lVert\mathbf T_k\rVert\le\sigma_k$ | `cp.norm(T, axis=1) <= sigma` |
| $\sum_k \sigma_k\Delta t$ | `cp.Minimize(cp.sum(sigma) * dt)` |

Plná verze téhle ukázky, ze které notebook vychází, je ve skriptu [`kod/ukazka_raketa.py`](https://github.com/tomasvicar/OMM-public/blob/master/cviceni/C1/kod/ukazka_raketa.py) v repozitáři předmětu.

In [ ]:
try:
    import cvxpy as cp
except ImportError:
    %pip install -q cvxpy
    import cvxpy as cp

In [ ]:
import cvxpy as cp
import matplotlib.pyplot as plt
import numpy as np

## Model a řešení

In [ ]:
N, dt = 60, 1.0                       # počet kroků a délka kroku [s]
g, m = 9.81, 2000.0                   # tíhové zrychlení [m/s2], hmotnost rakety [kg]
gv = np.array([0.0, g])
Tmax = 30000.0  # @param {type:"slider", min:20000, max:45000, step:1000}
Tmin = 5000.0                         # motor nejde vypnout ani škrtit pod minimum
p0 = np.array([1200.0, 1500.0])       # počáteční poloha [m]
v0 = np.array([-60.0, -80.0])         # počáteční rychlost [m/s]

p = cp.Variable((N + 1, 2))
v = cp.Variable((N + 1, 2))
T = cp.Variable((N, 2))
sigma = cp.Variable(N)                # pomocná horní mez velikosti tahu

omezeni = [p[0] == p0, v[0] == v0, p[N] == 0, v[N] == 0,   # start a měkké dosednutí
           p[:, 1] >= 0.5 * cp.abs(p[:, 0]),               # přistávací kužel
           cp.norm(T, axis=1) <= sigma, sigma >= Tmin, sigma <= Tmax]
for k in range(N):                    # diskretizovaná Newtonova rovnice
    omezeni += [v[k + 1] == v[k] + dt * (T[k] / m - gv),
                p[k + 1] == p[k] + dt * (v[k] + v[k + 1]) / 2]

uloha = cp.Problem(cp.Minimize(cp.sum(sigma) * dt), omezeni)
uloha.solve()
tah = np.linalg.norm(T.value, axis=1)

print(f"stav řešení: {uloha.status}")
print(f"spotřebovaný impuls: {uloha.value / 1e6:.2f} MN*s")
print(f"velikost tahu: {tah.min():.0f} až {tah.max():.0f} N (meze {Tmin:.0f}-{Tmax:.0f})")

## Kontrola, která umí selhat

Solver vrátil rovnou trajektorii i tahy, jenže trajektorie je jeho vlastní
proměnná — kdyby byla dynamika zapsaná špatně, vyšlo by to „správně“ i tak.
Nezávislá kontrola proto vezme **jen nalezené tahy**, odsimuluje s nimi let od
startu bez solveru a podívá se, kde raketa doopravdy skončí. Zároveň se ověří
i to, že se nikde neporušila spodní mez tahu, kvůli které se úloha
konvexifikovala.

In [ ]:
v_sim = np.vstack([v0, v0 + dt * np.cumsum(T.value / m - gv, axis=0)])
p_sim = np.vstack([p0, p0 + dt * np.cumsum((v_sim[:-1] + v_sim[1:]) / 2, axis=0)])
print(f"koncová poloha {np.linalg.norm(p_sim[-1]):.2e} m, "
      f"koncová rychlost {np.linalg.norm(v_sim[-1]):.2e} m/s")
assert np.linalg.norm(p_sim[-1]) < 1e-2 and np.linalg.norm(v_sim[-1]) < 1e-2

print(f"nejmenší tah {tah.min():.0f} N vs. minimum motoru {Tmin:.0f} N")
assert tah.min() >= Tmin - 1e-3, "lossless convexification by tady selhala"

## Obrázek

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))
ax1.plot(p_sim[:, 0], p_sim[:, 1], "-o", ms=3, label="trajektorie")
ax1.set(xlabel="vodorovná vzdálenost [m]", ylabel="výška [m]", title="Sestup rakety")
ax1.legend()
ax2.step(np.arange(N) * dt, tah / 1000, where="post", label="velikost tahu")
ax2.set(xlabel="čas [s]", ylabel="tah [kN]", title="Optimální řízení motoru")
ax2.legend()
plt.show()

## Na co se zeptat kódu

1. Průběh tahu vyšel *bang-bang* (maximum – minimum – maximum), nikdo to tak
   nezadal. Zkuste minimalizovat `cp.sum_squares(sigma)` místo `cp.sum(sigma)` —
   co se s profilem stane a co to fyzikálně znamená?
2. Stáhněte $T_{\max}$ posuvníkem pod zhruba 1,3násobek váhy rakety
   ($mg = 19{,}6$ kN): solver vrátí `infeasible`, tedy důkaz, že žádné takové
   řízení neexistuje — a to je taky výsledek, ne chyba.
3. Vyhoďte omezení přistávacího kužele. Účelová funkce vyjde *lépe* — proč, a jak
   by se bez znalosti správné odpovědi poznalo, že se počítá jiná úloha?